# lab_6A2F801A：日誌去識別化

## 任務一：去識別化技術實作 「遮罩」(Masking)

1.  假設有一個日誌檔案 ```web.log```，內容如下：

    ```bash
    2025-06-04 08:32:10 user=王小明 email=ming@example.com action=login ip=203.0.113.5
    2025-06-04 08:45:22 user=陳美華 email=meihua.chen@example.com action=reset_password ip=198.51.100.88
    2025-06-04 09:01:07 user=張志強 email=chiang@example.com action=view_medical_record patient_id=PT10029
    2025-06-04 09:15:44 user=李芳瑜 email=fangyu@example.com action=logout ip=203.0.113.99
    2025-06-04 09:33:19 user=王小明 email=ming@example.com action=download_report report_id=RPT-422 ip=192.168.1.200
    ```

1.  遮罩會部分隱藏資料，例如 ```ming@example.com``` 變成 ```m***@example.com```

1.  以下是一段 Python 代碼，執行遮蔭的工作。你可以在 Jupyter notebook 中執行，看看結果是怎樣。

In [ ]:
import re

def mask_email(email):
    # 保留第一個字母和 @ 之後的網域，中間用 *** 取代
    return re.sub(r'(^.)[^@]*(@.*)$', r'\1***\2', email)

def mask_ip(ip):
    # 保留前三碼，最後一碼設為 0（或 xxx）
    parts = ip.split('.')
    if len(parts) == 4:
        return f"{parts[0]}.{parts[1]}.{parts[2]}.0"
    return ip

input_file = '''
2025-06-04 08:32:10 user=王小明 email=ming@example.com action=login ip=203.0.113.5
2025-06-04 08:45:22 user=陳美華 email=meihua.chen@example.com action=reset_password ip=198.51.100.88
2025-06-04 09:01:07 user=張志強 email=chiang@example.com action=view_medical_record patient_id=PT10029
2025-06-04 09:15:44 user=李芳瑜 email=fangyu@example.com action=logout ip=203.0.113.99
2025-06-04 09:33:19 user=王小明 email=ming@example.com action=download_report report_id=RPT-422 ip=192.168.1.200
'''

result = ""

for line in input_file.splitlines():
    # 遮罩 email
    line = re.sub(r'[\w\.-]+@[\w\.-]+', lambda m: mask_email(m.group(0)), line)
    # 遮罩 IP 最後一段
    line = re.sub(r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b', lambda m: mask_ip(m.group(0)), line)
    result += line + "\n"

print(result)


4.  在新的日誌中，你看到那幾個欄位被「遮罩」？

5.  同學們，能夠將程式更改，將名字也「遮罩」嗎？  

    ```bash
    2025-06-04 08:32:10 user=*** email=m***@example.com action=login ip=203.0.113.0
    2025-06-04 08:45:22 user=*** email=m***@example.com action=reset_password ip=198.51.100.0
    2025-06-04 09:01:07 user=*** email=c***@example.com action=view_medical_record patient_id=PT10029
    2025-06-04 09:15:44 user=*** email=f***@example.com action=logout ip=203.0.113.0
    2025-06-04 09:33:19 user=*** email=m***@example.com action=download_report report_id=RPT-422 ip=192.168.1.0
    ```

6.  做法很簡單，只要在程式中，加上這句指令。請在以下的 Jupyter notebook 中，加入這句試試看？

    ```python
    line = re.sub(r'user=([^\s]+)', lambda m: f"user=***", line)
    ```

In [ ]:
import re

def mask_email(email):
    # 保留第一個字母和 @ 之後的網域，中間用 *** 取代
    return re.sub(r'(^.)[^@]*(@.*)$', r'\1***\2', email)

def mask_ip(ip):
    # 保留前三碼，最後一碼設為 0（或 xxx）
    parts = ip.split('.')
    if len(parts) == 4:
        return f"{parts[0]}.{parts[1]}.{parts[2]}.0"
    return ip

input_file = '''
2025-06-04 08:32:10 user=王小明 email=ming@example.com action=login ip=203.0.113.5
2025-06-04 08:45:22 user=陳美華 email=meihua.chen@example.com action=reset_password ip=198.51.100.88
2025-06-04 09:01:07 user=張志強 email=chiang@example.com action=view_medical_record patient_id=PT10029
2025-06-04 09:15:44 user=李芳瑜 email=fangyu@example.com action=logout ip=203.0.113.99
2025-06-04 09:33:19 user=王小明 email=ming@example.com action=download_report report_id=RPT-422 ip=192.168.1.200
'''

result = ""

for line in input_file.splitlines():
    # 遮罩 email
    line = re.sub(r'[\w\.-]+@[\w\.-]+', lambda m: mask_email(m.group(0)), line)
    # 遮罩 IP 最後一段
    line = re.sub(r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b', lambda m: mask_ip(m.group(0)), line)
    # 遮罩使用者名稱

    #
    result += line + "\n"

print(result)

## 任務二：去識別化技術實作 「哈希」(Hashing)

1.  「哈希」是單向函數，無法還原原始資料

1.  以下是一段 Python 代碼，執行「哈希」的工作。你可以在 Jupyter notebook 中執行，看看結果是怎樣。


In [ ]:
import hashlib
import re

def hash_value(value, salt=""):
    # 加鹽增加安全性（避免彩虹表攻擊）
    to_hash = value + salt
    return hashlib.sha256(to_hash.encode()).hexdigest()[:16]  # 取前16位節省空間

input_file = '''
2025-06-04 08:32:10 user=王小明 email=ming@example.com action=login ip=203.0.113.5
2025-06-04 08:45:22 user=陳美華 email=meihua.chen@example.com action=reset_password ip=198.51.100.88
2025-06-04 09:01:07 user=張志強 email=chiang@example.com action=view_medical_record patient_id=PT10029
2025-06-04 09:15:44 user=李芳瑜 email=fangyu@example.com action=logout ip=203.0.113.99
2025-06-04 09:33:19 user=王小明 email=ming@example.com action=download_report report_id=RPT-422 ip=192.168.1.200
'''

SALT = "school_lab_salt_2025"   # 實際環境應使用隨機且保密的值

for line in input_file.splitlines():
    # 對使用者名稱進行雜湊
    line = re.sub(r'user=([^\s]+)', lambda m: f"user={hash_value(m.group(1), SALT)}", line)
    print(line)

3.  在新的日誌中，你看到那幾個欄位變成「哈希值」？

4.  但可用於比對同一使用者的行為（例如同一 email 是否重複出現）

5.  同學們，能夠將程式更改，將email欄位也變成「哈希值」嗎？  

    ```bash
    2025-06-04 08:32:10 user=ff8a9360d3cd765a email=6dce094e3edbf916 action=login ip=203.0.113.5
    2025-06-04 08:45:22 user=51e20155dc05d594 email=2c5370d2c9b6a2a8 action=reset_password ip=198.51.100.88
    2025-06-04 09:01:07 user=9aaff092e4ad5c75 email=2d982b3b6db940a1 action=view_medical_record patient_id=PT10029
    2025-06-04 09:15:44 user=eb3f56d27959db31 email=c808e46041ff6035 action=logout ip=203.0.113.99
    2025-06-04 09:33:19 user=ff8a9360d3cd765a email=6dce094e3edbf916 action=download_report report_id=RPT-422 ip=192.168.1.200
    ```

6.  做法很簡單，只要在程式中，加上這句指令。請在以下的 Jupyter notebook 中，加入這句試試看？

    ```python
    line = re.sub(r'[\w\.-]+@[\w\.-]+', lambda m: hash_value(m.group(0), SALT), line)
    ```

In [ ]:
import hashlib
import re

def hash_value(value, salt=""):
    # 加鹽增加安全性（避免彩虹表攻擊）
    to_hash = value + salt
    return hashlib.sha256(to_hash.encode()).hexdigest()[:16]  # 取前16位節省空間

input_file = '''
2025-06-04 08:32:10 user=王小明 email=ming@example.com action=login ip=203.0.113.5
2025-06-04 08:45:22 user=陳美華 email=meihua.chen@example.com action=reset_password ip=198.51.100.88
2025-06-04 09:01:07 user=張志強 email=chiang@example.com action=view_medical_record patient_id=PT10029
2025-06-04 09:15:44 user=李芳瑜 email=fangyu@example.com action=logout ip=203.0.113.99
2025-06-04 09:33:19 user=王小明 email=ming@example.com action=download_report report_id=RPT-422 ip=192.168.1.200
'''

SALT = "school_lab_salt_2025"   # 實際環境應使用隨機且保密的值

for line in input_file.splitlines():
    # 對使用者名稱變成「哈希值」
    line = re.sub(r'user=([^\s]+)', lambda m: f"user={hash_value(m.group(1), SALT)}", line)
    # 對 email 變成「哈希值」

    #
    print(line)